In [ ]:
!pip install pypdf sentence-transformers faiss-cpu google-generativeai groq -q

In [ ]:
import faiss
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from groq import Groq
from google.colab import userdata

In [ ]:
pdf_path = "/content/drive/MyDrive/Colab Notebooks/Psychology2e_WEB.pdf"
reader = PdfReader(pdf_path)

# Chapter 8 Memory, Chapter 12 Social psychology and Chapter 15 Psychological disorders
pages_to_use = list(range(246, 277)) + list(range(398, 444)) + list(range(536, 598))

documents = []
for page_num in pages_to_use:
    text = reader.pages[page_num].extract_text()
    if text.strip():
        documents.append({
            "text": text,
            "page": page_num + 1
        })

In [ ]:
def split_into_chunks(documents, chunk_size=1000, overlap=200):
    chunks = []

    for doc in documents:
        text = doc["text"]
        page = doc["page"]

        start = 0
        while start < len(text):
            end = start + chunk_size
            chunk_text = text[start:end]

            if chunk_text.strip():
                chunks.append({
                    "text": chunk_text,
                    "page": page
                })

            start = end - overlap

    return chunks

chunks = split_into_chunks(documents)

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

chunk_texts = [chunk["text"] for chunk in chunks]
embeddings = model.encode(chunk_texts, show_progress_bar=True)

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
embeddings_array = np.array(embeddings).astype('float32')

dimension = embeddings_array.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings_array)

In [ ]:
def retrieve(query, top_k=3):
    q_embed = model.encode([query])
    q_embed = np.array(q_embed).astype('float32')

    distances, indices = index.search(q_embed, top_k)

    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "text": chunks[idx]["text"],
            "page": chunks[idx]["page"],
            "distance": distances[0][i]
        })

    return results

In [ ]:
results = retrieve("What is memory?")

for i, r in enumerate(results):
    print(f"Result {i+1} (page {r['page']}, distance: {r['distance']:.2f})")
    print(r["text"][:300] + "\n")

Result 1 (page 259, distance: 0.67)
es of these people in your house?
Wait . . . is this even your house? Uh oh, your stomach begins to rumble and you feel hungry. You’d like
something to eat, but you don’t know where the food is kept or even how to prepare it. Oh dear, this is getting
confusing. Maybe it would be best just go back to

Result 2 (page 260, distance: 0.72)
8.1 How Memory Functions
LEARNING OBJECTIVES
By the end of this section, you will be able to:
• Discuss the three basic functions of memory
• Describe the three stages of memory storage
• Describe and distinguish between procedural and declarative memory and semantic and episodic memory
Memory is an

Result 3 (page 262, distance: 0.75)
s, which we do not view as valuable information, we discard. If
we view something as valuable, the information will move into our short-term memory system.
Short-Term Memory
Short-term memory (STM) is a temporary storage system that processes incoming sensory memory. The
terms short-term

In [ ]:
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

client = Groq(api_key=GROQ_API_KEY)

In [ ]:
def ask(question, top_k=3):
    retrieved = retrieve(question, top_k)

    context = ""
    source_pages = []
    for chunk in retrieved:
        context += chunk["text"] + "\n\n"
        source_pages.append(chunk["page"])

    prompt = f"""Answer based on this context. If the answer isn't in the context, say so.

Context:{context}

Question:{question}
Answer:"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    answer = response.choices[0].message.content

    return {
        "answer": answer,
        "sources": source_pages
    }

In [ ]:
result = ask("What is short-term memory?")

print("Answer:", result["answer"])
print("Source pages:", result["sources"])

Answer: Short-term memory (STM) is a temporary storage system that processes incoming sensory memory. It takes information from sensory memory and connects that memory to something already stored, allowing it to be stored in long-term memory.
Source pages: [262, 262, 264]


In [ ]:
def ir_search(query, top_k=3):
    query_words = set(query.lower().split())

    scored = []
    for i, chunk in enumerate(chunks):
        chunk_words = set(chunk["text"].lower().split())

        matches = len(query_words.intersection(chunk_words))

        scored.append({
            "text": chunk["text"],
            "page": chunk["page"],
            "score": matches
        })

    scored.sort(key=lambda x: x["score"], reverse=True)

    return scored[:top_k]

In [ ]:
results = ir_search("What is short-term memory?")

for i, r in enumerate(results):
    print(f"Result {i+1} (page {r['page']}, matches: {r['score']})")
    print(r["text"][:300] + "\n")

Result 1 (page 262, matches: 3)
f
memory supervises or controls the flow of information to and from the three short-term systems, and the
central executive is responsible for moving information into long-term memory.
Sensory Memory
In the Atkinson-Shiffrin model, stimuli from the environment are processed first in sensory memory: 

Result 2 (page 263, matches: 3)
s less
activated over time, and the information is forgotten. However, Keppel and Underwood (1962) examined only
the first trials of the trigram task and found that proactive interference also affected short-term memory
retention. During proactive interference, previously learned information interfe

Result 3 (page 272, matches: 3)
vies such as the Jason Bourne spy thrillers. However, for real-life sufferers of
retrograde amnesia, like former NFL football player Scott Bolzan, the story is not a Hollywood movie. Bolzan
fell, hit his head, and deleted 46 years of his life in an instant. He is now living with one of the most extr

In [ ]:
def ask_llm(question):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": question}]
    )

    return response.choices[0].message.content

In [ ]:
llm_answer = ask_llm("What is short-term memory?")
print("LLM answer:", llm_answer)

LLM answer: Short-term memory is a temporary storage system in the brain that holds information for a short period, typically ranging from a few seconds to a minute, depending on the individual. It's a fundamental component of working memory, which is responsible for processing and manipulating information in the present moment.

Short-term memory has several key characteristics:

1. **Limited capacity**: Short-term memory can only hold a limited amount of information, usually around 7 ± 2 chunks of information (e.g., phone numbers, passwords, or a short list of items).
2. **Duration**: Information in short-term memory lasts for a short period, usually seconds to minutes.
3. **Voluntary nature**: Short-term memory is controlled by the individual's attention and intention to remember specific information.
4. **Rehearsal-sensitive**: Information in short-term memory can be rehearsed and repeated to help maintain it in memory for a longer period.

The process of placing information in sho

In [ ]:
def compare_all(question):
    print(f"Question: {question}")

    rag_result = ask(question)
    print(f"RAG Answer: {rag_result['answer']}")
    print(f"RAG Source pages: {rag_result['sources']}\n")

    ir_results = ir_search(question)
    print(f"IR top result (page {ir_results[0]['page']}):")
    print(ir_results[0]["text"][:200] + "\n")

    llm_answer = ask_llm(question)
    print(f"LLM answer: {llm_answer}")

In [ ]:
compare_all("What are the stages of memory?")

Question: What are the stages of memory?
RAG Answer: The stages of memory, according to Atkinson and Shiffrin's model, are:

1. Sensory Memory
2. Short-Term Memory
3. Long-Term Memory
RAG Source pages: [262, 262, 260]

IR top result (page 254):
l situations and routines of daily behavior.
7.2 Language
Language is a communication system that has both a lexicon and a system of grammar. Language acquisition
occurs naturally and effortlessly dur

LLM answer: The stages of memory refer to the process by which information is encoded, stored, and retrieved in the brain. Here are the three primary stages of memory, as described by the multi-store model developed by psychologist Atkinson and Shiffrin:

1. **Sensory Memory (0-2 seconds)**: This is the initial stage of memory, where information is briefly stored in the sensory buffers of the brain. It lasts for a very short period, often referred to as the "window of opportunity" for information to be encoded into the next stage.

2. **Short-Term

In [ ]:
queries_with_answers = [
    "What is short-term memory?",
    "What are the stages of memory?",
    "What is classical conditioning?",
    "What are the Big Five personality traits?",
    "What is schizophrenia?",
    "What is the difference between retrograde and anterograde amnesia?",
    "What is Freud's theory of personality?"
]
queries_no_answer = [
    "What is photosynthesis?",
    "How does the stock market work?"
]
query_unrelated = "What is the capital of France?"

In [ ]:
print("QUERIES WITH EXPECTED ANSWERS")

for i, q in enumerate(queries_with_answers, 1):
    print(f"Query{i}: {q}")
    result = ask(q)
    print(f"Answer: {result['answer'][:500]}...")
    print(f"Sources: pages {result['sources']}")

QUERIES WITH EXPECTED ANSWERS
Query1: What is short-term memory?
Answer: Short-term memory is a temporary storage system that processes incoming sensory memory. It is a component of working memory, where information from sensory memory is processed and sometimes connected to existing memories in order to be stored in long-term memory....
Sources: pages [262, 262, 264]
Query2: What are the stages of memory?
Answer: According to the provided context, the stages of memory are:

1. Sensory Memory
2. Short-Term Memory
3. Long-Term Memory

These stages were first proposed by Richard Atkinson and Richard Shiffrin (1968) in their model of human memory known as the Atkinson and Shiffrin's model....
Sources: pages [262, 262, 260]
Query3: What is classical conditioning?
Answer: Classical conditioning is a type of learning in which a neutral stimulus becomes associated with an unconditioned stimulus that provokes anxiety or distress, causing the neutral stimulus to elicit the same response as the 

In [ ]:
print("QUERIES WITH NO EXPECTED ANSWERS")

for i, q in enumerate(queries_no_answer, 1):
    print(f"Query{i}: {q}")
    result = ask(q)
    print(f"Answer: {result['answer']}")
    print(f"Sources: pages {result['sources']}")

QUERIES WITH NO EXPECTED ANSWERS
Query1: What is photosynthesis?
Answer: I do not have any information about photosynthesis in the provided context. This is a question about the context, it seems to be unrelated to the given text.
Sources: pages [265, 263, 262]
Query2: How does the stock market work?
Answer: This answer isn't found within the given context, which appears to be a passage about persuasion and the process of human memory.
Sources: pages [426, 277, 262]


In [ ]:
print("GENERAL UNRELATED QUERY")

print(f"Query: {query_unrelated}")

result = ask(query_unrelated)
print(f"Answer: {result['answer']}")
print(f"Sources: pages {result['sources']}")

GENERAL UNRELATED QUERY
Query: What is the capital of France?
Answer: I'm unable to provide the answer as it is not related to the given context. The context only talks about the prefrontal cortex and suicide rates, and does not mention France or its capital.
Sources: pages [267, 580, 405]
